# Método de Gauss-Seidel para sistemas de ecuaciones lineales

Implementación y aplicación del **método iterativo de Gauss-Seidel** para aproximar la solución de sistemas de ecuaciones lineales utilizando Python.

Este notebook reúne los códigos de Gauss-Seidel desarrollados originalmente junto con el estudio del método de Jacobi y los reorganiza como un ejercicio independiente. Se presentan dos implementaciones: mediante **matrices** y mediante **sumatorias**.

### Objetivos

- Comprender el funcionamiento del método de Gauss-Seidel.
- Implementar el método mediante una formulación matricial.
- Implementar el método directamente mediante sumatorias.
- Analizar el criterio de convergencia utilizado en la formulación matricial.
- Comparar los resultados obtenidos con ambas implementaciones.

**Tecnologías:** Python · NumPy

## 1. Fundamento del método

El **método de Gauss-Seidel** es un procedimiento iterativo utilizado para aproximar la solución de un sistema lineal:

$Ax=b$

A diferencia del método de Jacobi, Gauss-Seidel utiliza inmediatamente los valores que se van calculando dentro de la misma iteración.

Para cada componente, el método puede expresarse como:

$$
x_i^{(k)}=
\frac{1}{a_{ii}}
\left(
b_i
-
\sum_{j=1}^{i-1} a_{ij}x_j^{(k)}
-
\sum_{j=i+1}^{n} a_{ij}x_j^{(k-1)}
\right)
$$

Como criterio de parada se utiliza la diferencia entre dos aproximaciones consecutivas mediante la norma infinito:

$$
\left\|x^{(k)}-x^{(k-1)}\right\|_{\infty}<\text{Tol}
$$

### Sistema utilizado

Se desea aproximar la solución del sistema:

$$
\begin{aligned}
3x_1-x_2 &= 2 \\
-x_1+4x_2-x_3 &= 3 \\
-x_2+5x_3 &= 5
\end{aligned}
$$

con valor inicial:

$$
x^{(0)}=
\begin{pmatrix}
0\\
0\\
0
\end{pmatrix}
$$

En las implementaciones se utiliza una tolerancia de $10^{-6}$.

## 2. Implementación mediante matrices

En la formulación matricial se considera la descomposición:

$A=D-L-U$

donde $D$ contiene la diagonal de $A$, mientras que $L$ y $U$ representan las partes inferior y superior utilizadas por la implementación.

La iteración se expresa como:

$$
x^{(k+1)}=T_gx^{(k)}+C_g
$$

con:

$$
T_g=(D-L)^{-1}U
$$

y:

$$
C_g=(D-L)^{-1}b
$$

Antes de realizar las iteraciones se calculan los valores propios de $T_g$. El método continúa cuando el radio espectral cumple:

$$
\rho(T_g)<1
$$

El siguiente bloque corresponde al **código original** de la implementación mediante matrices.

In [1]:
"""El método de Jacobi matrices es muy similar a Gauss-Seidel ya que es el método iterativo es el mismo
el criterio de parada es el mismo, si converge o no se define por el radio espectral, la única diferencia
es la construcción de la matriz de transformación"""

import time
import numpy as np

matrizCoeficientes=np.array([[3,-1,0], [-1,4, -1], [0,-1,5]],float)
vectorIndependiente=np.array([2,3,5],float)
x0=np.zeros_like(vectorIndependiente)
tol=1e-6
nmax=50

def GaussSeidelMatrices(matrizCoeficientes, vectorIndependiente, x0, tol):
    D=np.diag(np.diag(matrizCoeficientes))
    U=D-np.triu(matrizCoeficientes)
    L=D-np.tril(matrizCoeficientes)
    Tg=np.dot(np.linalg.inv(D-L), U)
    print(Tg)
    Cg=np.dot(np.linalg.inv(D-L), vectorIndependiente)
    eig, eiv=np.linalg.eig(Tg) # Esta rutina retorna 2 parametros: (Valores propios y vectores propios)
    radio=max(abs(eig))
    
    if (radio<1):
        error=1
        iter = 0
        start = time.time()
        while (error>tol):
            x_new=np.dot(Tg,x0)+Cg
            iter += 1
            print(x_new, "en la iteración:", iter)
            error=max(abs(x_new-x0))
            x0=x_new
        end = time.time()
        print("El tiempo de ejecución es:", end - start)
        return x_new, iter
    else:
        print("El sistema no converge")
        return

GaussSeidelMatrices(matrizCoeficientes, vectorIndependiente, x0, tol)

[[0.         0.33333333 0.        ]
 [0.         0.08333333 0.25      ]
 [0.         0.01666667 0.05      ]]
[0.66666667 0.91666667 1.18333333] en la iteración: 1
[0.97222222 1.28888889 1.25777778] en la iteración: 2
[1.0962963  1.33851852 1.2677037 ] en la iteración: 3
[1.11283951 1.3451358  1.26902716] en la iteración: 4
[1.11504527 1.34601811 1.26920362] en la iteración: 5
[1.11533937 1.34613575 1.26922715] en la iteración: 6
[1.11537858 1.34615143 1.26923029] en la iteración: 7
[1.11538381 1.34615352 1.2692307 ] en la iteración: 8
[1.11538451 1.3461538  1.26923076] en la iteración: 9
El tiempo de ejecución es: 0.0009248256683349609


(array([1.11538451, 1.3461538 , 1.26923076]), 9)

### Resultado

La implementación mediante matrices converge hacia una solución aproximada cercana a:

$x \approx (1.1153845,\ 1.3461538,\ 1.2692308)^T$

Con la tolerancia utilizada, el proceso alcanza la aproximación en **9 iteraciones**.

## 3. Implementación mediante sumatorias

El mismo método puede implementarse directamente a partir de la fórmula de Gauss-Seidel.

Para calcular cada componente se utilizan:

- los valores **nuevos** ya calculados en la iteración actual cuando $j<i$;
- los valores de la **iteración anterior** cuando $j>i$.

El siguiente bloque corresponde al **código original** utilizado para esta implementación.

In [2]:
# Método de clase
import numpy as np
A=np.array([[3,-1,0], [-1,4, -1], [0,-1,5]],float)
b=np.array([2,3,5],float)
x0=np.zeros_like(b)

tol=1e-6
nmax=50

def gauss_sumas(A, b, x0, tol):
    error=1
    
    n=len(b)
    while (error>tol):
        x_new=np.zeros_like(b)
        for i in range (n):
            aux1=0
            aux2=0
            for j in range(0,i):
                aux1+=np.dot(A[i][j], x_new[j])
            for j in range (i+1, n):
                aux2+=np.dot(A[i][j], x0[j])
            x_new[i]=(b[i]-aux1-aux2)/A[i][i]
        error=max(abs(x0-x_new))
        x0=np.copy(x_new)
        print(x_new)
    return (x_new)
gauss_sumas(A,b,x0,tol)

[0.66666667 0.91666667 1.18333333]
[0.97222222 1.28888889 1.25777778]
[1.0962963  1.33851852 1.2677037 ]
[1.11283951 1.3451358  1.26902716]
[1.11504527 1.34601811 1.26920362]
[1.11533937 1.34613575 1.26922715]
[1.11537858 1.34615143 1.26923029]
[1.11538381 1.34615352 1.2692307 ]
[1.11538451 1.3461538  1.26923076]


array([1.11538451, 1.3461538 , 1.26923076])

### Resultado

La implementación mediante sumatorias genera la misma sucesión de aproximaciones que la implementación matricial y converge hacia:

$x \approx (1.1153845,\ 1.3461538,\ 1.2692308)^T$

Esto confirma que ambas formas representan el mismo método de Gauss-Seidel para el sistema estudiado.

## 4. Comparación de las implementaciones

Las dos implementaciones producen las mismas aproximaciones para el sistema utilizado.

La diferencia principal se encuentra en la forma de representar el procedimiento:

- **Matrices:** construye una matriz de iteración y permite analizar la convergencia mediante su radio espectral.
- **Sumatorias:** aplica directamente la fórmula componente a componente, utilizando de inmediato los valores recién calculados.

En ambos casos se emplea la norma infinito como criterio para determinar cuándo la diferencia entre iteraciones es suficientemente pequeña.

## 5. Conclusiones

A partir de las implementaciones desarrolladas se puede concluir que:

- El método de Gauss-Seidel permite aproximar la solución de sistemas de ecuaciones lineales mediante un proceso iterativo.
- Los valores calculados durante una iteración se utilizan inmediatamente para obtener los componentes siguientes.
- El método puede implementarse tanto mediante matrices como mediante sumatorias.
- En la formulación matricial, el radio espectral permite analizar la convergencia del proceso.
- Para el sistema estudiado, ambas implementaciones convergen hacia la misma solución aproximada.
- La tolerancia determina qué tan pequeña debe ser la diferencia entre iteraciones antes de detener el método.